# TP - Teoria de Compiladores
### Grupo: 4

###Integrantes
- **Diego Fabrizio Mucha Alvarez - u202317069**
- **Nicole Yessenia Vasquez Tinco - u202322884**
- **Alessandro Daniel Bravo Castillo - u202224501**
- **Linda Libertad Leon Quispe - u202322089**
- **Alessandro Elías Hesse Pulache - u202318347**

In [2]:
!pip install antlr4-python3-runtime==4.13.2

In [1]:
!curl -O https://www.antlr.org/download/antlr-4.13.2-complete.jar

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2089k  100 2089k    0     0  7903k      0 --:--:-- --:--:-- --:--:-- 7916k


In [ ]:
%%writefile Formas.g4
grammar Formas;

programa: instrucciones EOF;

instrucciones: instruccion*;

instruccion
    : asignacion
    | punto
    | recta
    | triangulo
    | cuadrado
    | repetir
    | trasladar
    | mostrar
    ;

asignacion: ID IGUAL expr ;

repetir: REPETIR ID VECES LPAREN instruccion RPAREN ;

punto: PUNTO ID LPAREN expr COMA expr RPAREN ;

recta
    : RECTA ID LPAREN ID COMA ID RPAREN
    | RECTA ID LPAREN expr COMA expr COMA expr COMA expr RPAREN
    ;

triangulo
    : TRIANGULO ID LPAREN ID COMA ID COMA ID RPAREN
    | TRIANGULO ID LPAREN expr COMA expr COMA expr COMA expr COMA expr COMA expr RPAREN
    ;

cuadrado
    : CUADRADO ID LPAREN ID COMA ID COMA ID COMA ID RPAREN
    | CUADRADO ID LPAREN expr COMA expr COMA expr COMA expr COMA expr COMA expr COMA expr COMA expr RPAREN
    ;

trasladar
    : TRASLADAR LPAREN ID COMA expr COMA expr RPAREN
    ;

mostrar
    : MOSTRAR LPAREN ID RPAREN
    ;

expr
    : expr op=(MULT | DIV) expr
    | expr op=(MAS | MENOS) expr
    | NUM
    | ID
    | LPAREN expr RPAREN
    ;

REPETIR   : 'repetir';
VECES     : 'veces';
PUNTO     : 'punto';
RECTA     : 'recta';
TRIANGULO : 'triangulo';
TRASLADAR : 'trasladar';
MOSTRAR   : 'mostrar';

IGUAL : '=';
COMA  : ',';
LPAREN: '(';
RPAREN: ')';

MAS   : '+';
MENOS : '-';
MULT  : '*';
DIV   : '/';

ID  : [a-zA-Z][a-zA-Z0-9]*;
NUM : [0-9]+ ('.' [0-9]+)?;

WS : [ \t\r\n]+ -> skip;

Overwriting Formas.g4


In [4]:
!java -jar antlr-4.13.2-complete.jar -Dlanguage=Python3 -visitor Formas.g4

In [ ]:
%%writefile Visitor.py

if __name__ is not None and "." in __name__:
    from .FormasParser import FormasParser
    from .FormasVisitor import FormasVisitor
else:
    from FormasParser import FormasParser
    from FormasVisitor import FormasVisitor


class Visitor(FormasVisitor):
    def __init__(self):
        self.variables = {}
        self.figuras = {}
        self.html = ""

    def getFiguras(self):
        return self.figuras

    def getHtml(self):
        return self.html

    def visitPrograma(self, ctx):
        return self.visit(ctx.instrucciones())

    def visitInstrucciones(self, ctx):
        for instruccion in ctx.instruccion():
            self.visit(instruccion)

    def visitRepetir(self, ctx):
        for i in range(int(ctx.ID().getText())):
            self.visit(instruccion)
        
    def visitAsignacion(self, ctx):
        nombre = ctx.ID().getText()
        valor = self.visit(ctx.expr())
        self.variables[nombre] = valor

    def visitPunto(self, ctx):
        nombre = ctx.ID().getText()
        x = self.visit(ctx.expr(0))
        y = self.visit(ctx.expr(1))
        self.figuras[nombre] = {"tipo": "punto","x": x,"y": y}

    def visitRecta(self, ctx):
        nombre = ctx.ID(0).getText()
        if len(ctx.ID()) == 3:
            p1 = ctx.ID(1).getText()
            p2 = ctx.ID(2).getText()
            punto1 = self.figuras[p1]
            punto2 = self.figuras[p2]
            self.figuras[nombre] = {
                "tipo": "recta",
                "x1": punto1["x"],
                "y1": punto1["y"],
                "x2": punto2["x"],
                "y2": punto2["y"]
            }

        else:
            self.figuras[nombre] = {
                "tipo": "recta",
                "x1": self.visit(ctx.expr(0)),
                "y1": self.visit(ctx.expr(1)),
                "x2": self.visit(ctx.expr(2)),
                "y2": self.visit(ctx.expr(3))
            }

    def visitTriangulo(self, ctx):

        nombre = ctx.ID(0).getText()
        if len(ctx.ID()) == 4:
            p1 = self.figuras[ctx.ID(1).getText()]
            p2 = self.figuras[ctx.ID(2).getText()]
            p3 = self.figuras[ctx.ID(3).getText()]
            self.figuras[nombre] = {
                "tipo": "triangulo",
                "puntos": [
                    (p1["x"], p1["y"]),
                    (p2["x"], p2["y"]),
                    (p3["x"], p3["y"])
                ]
            }

        else:
            self.figuras[nombre] = { 
                "tipo": "triangulo",
                "puntos": [
                    (self.visit(ctx.expr(0)), self.visit(ctx.expr(1))),
                    (self.visit(ctx.expr(2)), self.visit(ctx.expr(3))),
                    (self.visit(ctx.expr(4)), self.visit(ctx.expr(5)))
                ]
            }

    def visitCuadrado(self, ctx):
        nombre = ctx.ID(0).getText()
        if (len(ctx.ID()))

    def visitTrasladar(self, ctx):
        nombre = ctx.ID().getText()
        dx = self.visit(ctx.expr(0))
        dy = self.visit(ctx.expr(1))
        figura = self.figuras[nombre]
        if figura["tipo"] == "punto":
            figura["x"] += dx
            figura["y"] += dy
        elif figura["tipo"] == "recta":
            figura["x1"] += dx
            figura["y1"] += dy
            figura["x2"] += dx
            figura["y2"] += dy
        elif figura["tipo"] == "triangulo":
            nuevos = []
            for x, y in figura["puntos"]:
                nuevos.append((x + dx, y + dy))
            figura["puntos"] = nuevos

    def visitMostrar(self, ctx):
        nombre = ctx.ID().getText()
        figura = self.figuras[nombre]
        self.html += self.generarFigura(nombre, figura)

    def generarFigura(self, nombre, figura):
        codigo = ""
        if figura["tipo"] == "punto":
            x = figura["x"]
            y = figura["y"]
            codigo += f"""
            ctx.beginPath();
            ctx.arc({x}, {y}, 5, 0, 2 * Math.PI);
            ctx.fill();
            ctx.fillText("{nombre}({x},{y})", {x}+10, {y}+10);
            """

        elif figura["tipo"] == "recta":
            x1 = figura["x1"]
            y1 = figura["y1"]
            x2 = figura["x2"]
            y2 = figura["y2"]
            codigo += f"""
            ctx.beginPath();
            ctx.moveTo({x1}, {y1});
            ctx.lineTo({x2}, {y2});
            ctx.stroke();
            ctx.fillText("{nombre}", ({x1}+{x2})/2, ({y1}+{y2})/2 - 12);
            """
        elif figura["tipo"] == "triangulo":
            p = figura["puntos"]
            codigo += f"""
            ctx.beginPath();
            ctx.moveTo({p[0][0]}, {p[0][1]});
            ctx.lineTo({p[1][0]}, {p[1][1]});
            ctx.lineTo({p[2][0]}, {p[2][1]});
            ctx.closePath();
            ctx.stroke();
            ctx.fillText(
                "{nombre}",
                ({p[0][0]} + {p[1][0]} + {p[2][0]}) / 3,
                ({p[0][1]} + {p[1][1]} + {p[2][1]}) / 3
            );
            """
        return codigo

    def visitExpr(self, ctx):
        if ctx.NUM():
            n = ctx.NUM().getText()
            if "." in n:
                return float(n)
            else:
                return int(n)
        if ctx.ID():
            nombre = ctx.ID().getText()
            return self.variables[nombre]
        if ctx.getChildCount() == 3:
            if ctx.getChild(0).getText() == "(":
                return self.visit(ctx.expr(0))
            izquierda = self.visit(ctx.expr(0))
            derecha = self.visit(ctx.expr(1))
            operador = ctx.getChild(1).getText()
            if operador == "+":
                return izquierda + derecha
            elif operador == "-":
                return izquierda - derecha
            elif operador == "*":
                return izquierda * derecha
            elif operador == "/":
                return izquierda / derecha

Overwriting Visitor.py


In [6]:
%%writefile ejemplo1.geo
punto A(190,60)
x = 120
y = 30

punto B(x,y)
punto C(260,30)

recta r1(A,B)
triangulo t1(A,B,C)

mostrar(A)
mostrar(B)
mostrar(C)
mostrar(r1)
mostrar(t1)

Writing ejemplo1.geo


In [8]:
from antlr4 import *
from FormasLexer import FormasLexer
from FormasParser import FormasParser
from Visitor import Visitor

input_stream = FileStream('ejemplo1.geo')

lexer = FormasLexer(input_stream)
token_stream = CommonTokenStream(lexer)
parser = FormasParser(token_stream)

tree = parser.programa()

print(tree.toStringTree(recog=parser))

visitor = Visitor()
visitor.visit(tree)

print(visitor.getFiguras())

(programa (instrucciones (instruccion (punto punto A ( (expr 190) , (expr 60) ))) (instruccion (asignacion x = (expr 120))) (instruccion (asignacion y = (expr 30))) (instruccion (punto punto B ( (expr x) , (expr y) ))) (instruccion (punto punto C ( (expr 260) , (expr 30) ))) (instruccion (recta recta r1 ( A , B ))) (instruccion (triangulo triangulo t1 ( A , B , C ))) (instruccion (mostrar mostrar ( A ))) (instruccion (mostrar mostrar ( B ))) (instruccion (mostrar mostrar ( C ))) (instruccion (mostrar mostrar ( r1 ))) (instruccion (mostrar mostrar ( t1 )))) <EOF>)
{'A': {'tipo': 'punto', 'x': 190, 'y': 60}, 'B': {'tipo': 'punto', 'x': 120, 'y': 30}, 'C': {'tipo': 'punto', 'x': 260, 'y': 30}, 'r1': {'tipo': 'recta', 'x1': 190, 'y1': 60, 'x2': 120, 'y2': 30}, 't1': {'tipo': 'triangulo', 'puntos': [(190, 60), (120, 30), (260, 30)]}}


In [9]:
from IPython.display import HTML

html_content = f"""
<div style="padding: 20px; background-color: #f0f0f0; border-radius: 10px;">
  <h2 style="color: #333;">Interfaz de figuras geométricas</h2>

  <canvas id="canvas" width="500" height="500" style="background-color: white; border: 1px solid #999;"></canvas>

  <script>
    const canvas = document.getElementById("canvas");
    const ctx = canvas.getContext("2d");

    ctx.font = "12px Arial";
    ctx.lineWidth = 2;

    {visitor.getHtml()}
  </script>
</div>
"""

HTML(html_content)